# H1: Валидация определения слабых мест экспертом

**Цель:** сравнить слабые места, определённые алгоритмом, с независимой экспертной оценкой.

**Гипотеза:**
- H₀: Совпадение алгоритма и эксперта — случайное (κ < 0.2)
- H₁: Совпадение значимое (κ ≥ 0.4)

**Протокол:** см. `docs/INTERVIEW_EXPERT.md`

**Связь с методологией:** см. `docs/METHODOLOGY_FEATURES.md`, раздел 4

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score, confusion_matrix, classification_report
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')

CATEGORIES = ['farming', 'combat', 'survival', 'vision', 'objectives', 'mechanics', 'consistency', 'control']
CAT_NAMES = ['Фарм', 'Бой', 'Выживаемость', 'Вижн', 'Объекты', 'Механика', 'Стабильность', 'Контроль']

## 1. Ввод данных

Заполните оценки алгоритма и эксперта для каждого игрока.

**Формат:** словарь `{player_id: {category: score}}` для алгоритма и эксперта.

In [ ]:
# ===== ЗАПОЛНИТЬ ПОСЛЕ ИНТЕРВЬЮ =====

# Оценки АЛГОРИТМА (из API /ml/detailed-features/{account_id})
algo_scores = {
    'player_1': {'farming': 6.2, 'combat': 4.5, 'survival': 5.1, 'vision': 3.0, 'objectives': 4.8, 'mechanics': 5.5, 'consistency': 6.0, 'control': 2.5},
    'player_2': {'farming': 7.1, 'combat': 6.8, 'survival': 4.2, 'vision': 5.5, 'objectives': 6.0, 'mechanics': 7.0, 'consistency': 5.5, 'control': 4.0},
    # Добавьте остальных игроков...
}

# Оценки ЭКСПЕРТА (из анкеты, шкала 1-10)
expert_scores = {
    'player_1': {'farming': 6.0, 'combat': 5.0, 'survival': 5.0, 'vision': 4.0, 'objectives': 5.0, 'mechanics': 6.0, 'consistency': 6.0, 'control': 3.0},
    'player_2': {'farming': 7.0, 'combat': 7.0, 'survival': 4.0, 'vision': 6.0, 'objectives': 5.5, 'mechanics': 7.5, 'consistency': 6.0, 'control': 5.0},
    # Добавьте остальных игроков...
}

# Топ-3 слабых мест АЛГОРИТМА (ключи категорий)
algo_top3_weak = {
    'player_1': ['control', 'vision', 'combat'],
    'player_2': ['survival', 'control', 'consistency'],
}

# Топ-3 слабых мест ЭКСПЕРТА (из анкеты)
expert_top3_weak = {
    'player_1': ['vision', 'control', 'objectives'],
    'player_2': ['survival', 'consistency', 'control'],
}

print(f'Игроков в выборке: {len(algo_scores)}')

## 2. Корреляция оценок (по шкале 1-10)

In [ ]:
# Собрать все пары (algo_score, expert_score) по всем игрокам и категориям
pairs = []
for pid in algo_scores:
    if pid not in expert_scores:
        continue
    for cat in CATEGORIES:
        a = algo_scores[pid].get(cat, 5)
        e = expert_scores[pid].get(cat, 5)
        pairs.append({'player': pid, 'category': cat, 'algo': a, 'expert': e})

pairs_df = pd.DataFrame(pairs)

# Spearman correlation
rho, p = stats.spearmanr(pairs_df['algo'], pairs_df['expert'])
print(f'Корреляция Спирмена (все пары): ρ = {rho:.3f}, p = {p:.4f}')

# Pearson
r, p_pearson = stats.pearsonr(pairs_df['algo'], pairs_df['expert'])
print(f'Корреляция Пирсона:             r = {r:.3f}, p = {p_pearson:.4f}')

# MAD (Mean Absolute Difference)
mad = np.abs(pairs_df['algo'] - pairs_df['expert']).mean()
print(f'Средняя абсолютная разница:     {mad:.2f} баллов')

# Per-category correlation
print('\nКорреляция по категориям:')
for cat, name in zip(CATEGORIES, CAT_NAMES):
    sub = pairs_df[pairs_df['category'] == cat]
    if len(sub) < 3:
        continue
    r_cat, _ = stats.spearmanr(sub['algo'], sub['expert'])
    mad_cat = np.abs(sub['algo'] - sub['expert']).mean()
    print(f'  {name:<20} ρ={r_cat:>6.3f}  MAD={mad_cat:.2f}')

## 3. Cohen's Kappa: совпадение по слабым местам

In [ ]:
# Для каждого (player, category): бинарная метка — "слабое место" или нет
algo_labels = []
expert_labels = []

for pid in algo_top3_weak:
    if pid not in expert_top3_weak:
        continue
    for cat in CATEGORIES:
        algo_labels.append(1 if cat in algo_top3_weak[pid] else 0)
        expert_labels.append(1 if cat in expert_top3_weak[pid] else 0)

kappa = cohen_kappa_score(algo_labels, expert_labels)
print(f'Cohen\'s Kappa (слабое место / нет): κ = {kappa:.3f}')

interpretation = (
    'Хорошее согласие' if kappa > 0.6 else
    'Умеренное согласие' if kappa > 0.4 else
    'Слабое согласие' if kappa > 0.2 else
    'Случайное совпадение'
)
print(f'Интерпретация: {interpretation}')

# Confusion matrix
cm = confusion_matrix(expert_labels, algo_labels)
print(f'\nConfusion Matrix (expert \'слабое\' vs algo \'слабое\'):')
print(f'  TN={cm[0,0]}, FP={cm[0,1]}')
print(f'  FN={cm[1,0]}, TP={cm[1,1]}')

# Top-3 overlap
overlaps = []
for pid in algo_top3_weak:
    if pid not in expert_top3_weak:
        continue
    overlap = len(set(algo_top3_weak[pid]) & set(expert_top3_weak[pid]))
    overlaps.append(overlap)
    print(f'  {pid}: совпадение {overlap}/3 ({set(algo_top3_weak[pid]) & set(expert_top3_weak[pid])})')

print(f'\nСредний Top-3 Overlap: {np.mean(overlaps):.1f} из 3')

## 4. Визуализация

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: algo vs expert scores
axes[0].scatter(pairs_df['expert'], pairs_df['algo'], alpha=0.5, c='cyan', s=40)
axes[0].plot([0, 10], [0, 10], 'r--', alpha=0.5, label='Идеальное совпадение')
axes[0].set_xlabel('Оценка эксперта')
axes[0].set_ylabel('Оценка алгоритма')
axes[0].set_title(f'Оценки: алгоритм vs эксперт (ρ={rho:.2f})')
axes[0].legend()
axes[0].set_xlim(0, 10)
axes[0].set_ylim(0, 10)

# Heatmap: confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Не слабое', 'Слабое'], yticklabels=['Не слабое', 'Слабое'])
axes[1].set_xlabel('Алгоритм')
axes[1].set_ylabel('Эксперт')
axes[1].set_title(f'Confusion Matrix (κ={kappa:.2f})')

plt.tight_layout()
plt.savefig('H1_expert_validation.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Выводы

| Метрика | Значение | Интерпретация |
|---------|----------|---------------|
| Spearman ρ | ? | ? |
| Cohen's κ | ? | ? |
| Top-3 Overlap | ?/3 | ? |
| MAD | ? баллов | ? |

> **Заполняется после проведения интервью и запуска блокнота.**

### Интерпретация

- κ ≥ 0.4 → **H₁ подтверждена**: алгоритм адекватно определяет слабые места
- κ < 0.2 → **H₀ не отвергнута**: требуется пересмотр расчёта фичей
- Top-3 Overlap ≥ 2/3 → алгоритм совпадает с экспертом в ключевых рекомендациях